In [ ]:
# Cell 0 - Imports
import os
import glob
import random
import gc
import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import timm

from tqdm import tqdm
import copy
from torch.optim import optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Imports loaded successfully.")

In [ ]:
# Cell 1 - Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2 - Aggressive Social Media Compression Function
def apply_social_media_compression(img_bgr):
    """
    Simulates an aggressive social-media pipeline:
    Downscale -> Gaussian blur -> JPEG compression -> RGB
    """
    h, w, _ = img_bgr.shape

    # Step 1: Downscale and upscale
    scale_factor = random.uniform(0.8, 1.0)
    new_h, new_w = max(1, int(h * scale_factor)), max(1, int(w * scale_factor))

    img_resized = cv2.resize(
        img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA
    )
    img_resized = cv2.resize(
        img_resized, (w, h), interpolation=cv2.INTER_LINEAR
    )

    # Step 2: Mild blur
    if random.random() < 0.7:
        img_resized = cv2.GaussianBlur(img_resized, (5, 5), 0)

    # Step 3: JPEG compression
    quality = random.randint(10, 25)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    success, encimg = cv2.imencode(".jpg", img_resized, encode_param)

    if not success:
        raise RuntimeError("JPEG compression failed.")

    decimg_bgr = cv2.imdecode(encimg, 1)

    # Convert BGR -> RGB
    return cv2.cvtColor(decimg_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# Cell 3 - Balanced Dataset Class with Compression Support
class BalancedDeepfakeDataset(Dataset):
    def __init__(
        self,
        split="train",
        base_dir="/kaggle/input",
        transform=None,
        seed=42,
        compression_prob=0.5
    ):
        self.transform = transform
        self.compression_prob = compression_prob
        self.image_paths = []
        self.labels = []
        self.rng = random.Random(seed)

        self._temp_data = {
            "celeb": {0: [], 1: []},
            "140k": {0: [], 1: []},
            "ffpp": {0: [], 1: []}
        }

        print(f"[{split.upper()}] Scanning {base_dir} to find datasets...")
        dataset_paths = self._auto_discover_paths(base_dir)

        self._load_celebdf(dataset_paths["celeb"], split)
        self._load_140k(dataset_paths["140k"], split)
        self._load_ffpp(dataset_paths["ffpp"], split)

        self._balance_and_merge(split)

    def _auto_discover_paths(self, base_dir):
        paths = {"celeb": None, "ffpp": None, "140k": None}

        for root, dirs, files in os.walk(base_dir):
            if "Celeb_V2" in dirs and paths["celeb"] is None:
                paths["celeb"] = os.path.join(root, "Celeb_V2")

            if "dataset_processed_split" in dirs and paths["ffpp"] is None:
                paths["ffpp"] = os.path.join(root, "dataset_processed_split")

            if "real-vs-fake" in dirs and paths["140k"] is None:
                paths["140k"] = os.path.join(root, "real-vs-fake")

        return paths

    def _add_from_folder(self, dataset_name, folder_path, label):
        if os.path.exists(folder_path):
            for ext in ("*.png", "*.jpg", "*.jpeg"):
                paths = glob.glob(os.path.join(folder_path, ext))
                self._temp_data[dataset_name][label].extend(paths)

    def _load_celebdf(self, base_path, split):
        split_map = {"train": "Train", "val": "Val", "test": "Test"}
        target_split = split_map.get(split)

        if base_path and os.path.exists(base_path) and target_split:
            self._add_from_folder(
                "celeb", os.path.join(base_path, target_split, "real"), 0
            )
            self._add_from_folder(
                "celeb", os.path.join(base_path, target_split, "fake"), 1
            )

    def _load_140k(self, base_path, split):
        split_map = {"train": "train", "val": "valid", "test": "test"}
        target_split = split_map.get(split)

        if base_path and os.path.exists(base_path) and target_split:
            self._add_from_folder(
                "140k", os.path.join(base_path, target_split, "real"), 0
            )
            self._add_from_folder(
                "140k", os.path.join(base_path, target_split, "fake"), 1
            )

    def _load_ffpp(self, base_path, split):
        if base_path:
            split_path = os.path.join(base_path, split)

            if os.path.exists(split_path):
                for class_dir in os.listdir(split_path):
                    dir_path = os.path.join(split_path, class_dir)

                    if os.path.isdir(dir_path):
                        label = 0 if class_dir == "Real" else 1
                        self._add_from_folder(
                            "ffpp", dir_path, label
                        )

    def _balance_and_merge(self, split):
        min_count = float("inf")

        for ds_name, labels in self._temp_data.items():
            count_real = len(labels[0])
            count_fake = len(labels[1])

            min_count = min(
                min_count,
                count_real,
                count_fake
            )

        if min_count == 0:
            print(
                f"ERROR: One or more categories in "
                f"{split.upper()} has 0 images. Cannot balance."
            )
            return

        print(
            f"[{split.upper()}] Balancing to "
            f"{min_count} images per category..."
        )

        for ds_name, labels in self._temp_data.items():
            for label, paths in labels.items():
                sampled_paths = self.rng.sample(paths, min_count)

                self.image_paths.extend(sampled_paths)
                self.labels.extend([label] * min_count)

        combined = list(zip(self.image_paths, self.labels))

        if not combined:
            return

        self.rng.shuffle(combined)

        self.image_paths, self.labels = zip(*combined)
        self.image_paths = list(self.image_paths)
        self.labels = list(self.labels)

        print(
            f"Final {split} dataset size: "
            f"{len(self.image_paths)} images ready.\n"
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        img_bgr = cv2.imread(img_path)

        if img_bgr is None:
            raise RuntimeError(f"Could not read image: {img_path}")

        if self.rng.random() < self.compression_prob:
            img_rgb = apply_social_media_compression(img_bgr)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        img_pil = Image.fromarray(img_rgb)

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        return img_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# Cell 4 - Define Transforms for Xception (299x299)
def get_data_transforms(img_size=299):
    norm_mean = [0.485, 0.456, 0.406]
    norm_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(
            img_size,
            padding=4,
            padding_mode="reflect"
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.1,
            hue=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=norm_mean,
            std=norm_std
        )
    ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=norm_mean,
            std=norm_std
        )
    ])

    return train_transform, val_transform

train_transform, val_transform = get_data_transforms(img_size=299)

In [ ]:
# Cell 5 - Initialize Datasets and DataLoaders
# Compression is used only for training.
# Validation and test data remain clean.

train_dataset = BalancedDeepfakeDataset(
    split="train",
    transform=train_transform,
    seed=42,
    compression_prob=0.7
)

val_dataset = BalancedDeepfakeDataset(
    split="val",
    transform=val_transform,
    seed=42,
    compression_prob=0.0
)

test_dataset = BalancedDeepfakeDataset(
    split="test",
    transform=val_transform,
    seed=42,
    compression_prob=0.0
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Cell 6 - FINAL Xception Model
# IMPORTANT:
# This is the SAME model that will be:
# 1) wrapped by ART,
# 2) adversarially trained,
# 3) evaluated,
# 4) used for Grad-CAM.

class SpatialDeepfakeDetector(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model(
            "xception",
            pretrained=pretrained
        )

        num_ftrs = self.backbone.fc.in_features

        # Two logits: class 0 = Real, class 1 = Fake
        self.backbone.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.backbone(x)


print("Loading final Xception model...")
model = SpatialDeepfakeDetector(pretrained=True).to(device)

print("Model initialized.")

In [ ]:
# Cell 7 - Verify Final Model
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print(f"Batch images shape : {images.shape}")
print(f"Batch labels shape : {labels.shape}")
print(f"Model outputs shape: {outputs.shape}")

assert outputs.shape[1] == 2, "Model must output 2 logits."
print("Model output check: PASSED")

In [ ]:
# Cell 8 - Inspect One Batch
images, labels = next(iter(train_loader))

print(f"Batch images tensor shape: {images.shape}")
print(f"Batch labels tensor shape: {labels.shape}")
print(f"Batch labels: {labels[:16].tolist()}")

In [ ]:
# Cell 9 - Visualization Imports
import matplotlib.pyplot as plt
import torchvision

print("Visualization tools ready.")

In [ ]:
# Cell 10 - Image Display Helper
def imshow(inp, title=None):
    inp = inp.detach().cpu().numpy().transpose((1, 2, 0))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)

    plt.figure(figsize=(12, 8))
    plt.imshow(inp)

    if title is not None:
        plt.title(title)

    plt.axis("off")
    plt.show()

In [ ]:
# Cell 11 - Visualize Training Samples
images, labels = next(iter(train_loader))

num_images_to_show = min(8, images.size(0))

images_to_show = images[:num_images_to_show]
labels_to_show = labels[:num_images_to_show]

out = torchvision.utils.make_grid(
    images_to_show,
    nrow=4
)

label_mapping = {
    0: "Real",
    1: "Fake"
}

titles = [
    label_mapping[int(label.item())]
    for label in labels_to_show
]

imshow(
    out,
    title=f"Sample Labels: {titles}"
)

In [ ]:
# Cell 12 - Install and Import ART
# Run this cell once. If ART is already installed, the install is harmless.

!pip install adversarial-robustness-toolbox -q

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

print("ART imported successfully.")

In [ ]:
# Cell 13 - Training Loss
# Because the final Xception model has two outputs:
# 0 = Real, 1 = Fake
# CrossEntropyLoss is used.

criterion = nn.CrossEntropyLoss()

print("Loss function:", criterion)

In [ ]:
# Cell 14 - Training Optimizer
# The optimizer belongs to the FINAL model.
# Do not create a second model after this point.

backbone_params = []
classifier_params = []

for name, param in model.named_parameters():
    if "backbone.fc" in name:
        classifier_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {
        "params": backbone_params,
        "lr": 1e-5
    },
    {
        "params": classifier_params,
        "lr": 1e-3
    }
])

print("Optimizer created for the final model.")

In [ ]:
# Cell 15 - Early Stopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose

        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False
        self.best_model_wts = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            if self.verbose:
                print(
                    f"Validation loss decreased "
                    f"({self.best_loss:.4f} --> {val_loss:.4f}). "
                    f"Saving best weights..."
                )

            self.best_loss = val_loss
            self.best_model_wts = copy.deepcopy(
                model.state_dict()
            )
            self.counter = 0

        else:
            self.counter += 1

            print(
                f"EarlyStopping counter: "
                f"{self.counter} out of {self.patience}"
            )

            if self.counter >= self.patience:
                self.early_stop = True

print("Early stopping class ready.")

In [ ]:
# Cell 16 - Create ART Classifier + FGSM/PGD Attacks
# IMPORTANT:
# ART wraps the SAME final model used by the training loop.
# No separate/different model is created here.

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ART expects a valid input range.
# Images are ImageNet-normalized, so use a range that contains
# the normalized [0,1] pixel range for all channels.
ART_CLIP_MIN = -2.1179
ART_CLIP_MAX = 2.6400

art_optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

classifier = PyTorchClassifier(
    model=model,
    clip_values=(ART_CLIP_MIN, ART_CLIP_MAX),
    loss=criterion,
    optimizer=art_optimizer,
    input_shape=(3, 299, 299),
    nb_classes=2,
    device_type="gpu" if torch.cuda.is_available() else "cpu"
)

# FGSM: one-step attack
attack_fgsm = FastGradientMethod(
    estimator=classifier,
    eps=0.05
)

# PGD: iterative attack
attack_pgd = ProjectedGradientDescent(
    estimator=classifier,
    eps=0.05,
    max_iter=10
)

print("ART classifier created successfully.")
print("FGSM attack created.")
print("PGD attack created.")
print("ART is connected to the FINAL Xception model.")

In [ ]:
# Cell 17 - ADVERSARIAL TRAINING
# This is the key cell that makes the notebook match the diagram.
#
# Clean training batch
#        |
#        +------> clean samples
#        |
#        +--> ART FGSM --> adversarial samples
#                         |
#                         v
#                 combine clean + adversarial
#                         |
#                         v
#                     Xception
#                         |
#                         v
#                       Loss
#                         |
#                         v
#                     Backprop
#
# We use FGSM for adversarial training to keep computation practical.
# PGD is used later for stronger robustness evaluation.

epochs = 8
early_stopper = EarlyStopping(
    patience=3,
    verbose=True
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=2,
    factor=0.1
)

for epoch in range(epochs):

    # =====================================================
    # TRAINING
    # =====================================================
    model.train()

    running_loss = 0.0
    train_correct = 0
    train_total = 0

    current_lr = optimizer.param_groups[0]["lr"]

    loop = tqdm(
        train_loader,
        total=len(train_loader),
        leave=True,
        miniters=100
    )

    for inputs, labels in loop:

        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()

        # -------------------------------------------------
        # 1. Generate adversarial examples with ART
        # -------------------------------------------------
        #
        # Attack generation is performed in evaluation mode
        # so dropout/batchnorm do not introduce extra randomness.
        model.eval()

        inputs_np = (
            inputs.detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        labels_np = (
            labels.detach()
            .cpu()
            .numpy()
            .astype(np.int64)
        )

        adv_inputs_np = attack_fgsm.generate(
            x=inputs_np,
            y=labels_np
        )

        adv_inputs = torch.from_numpy(
            adv_inputs_np
        ).float().to(device)

        # -------------------------------------------------
        # 2. Create adversarially augmented training batch
        # -------------------------------------------------
        combined_inputs = torch.cat(
            [inputs, adv_inputs],
            dim=0
        )

        combined_labels = torch.cat(
            [labels, labels],
            dim=0
        )

        # -------------------------------------------------
        # 3. Train on CLEAN + ADVERSARIAL data
        # -------------------------------------------------
        model.train()

        optimizer.zero_grad(set_to_none=True)

        outputs = model(combined_inputs)

        loss = criterion(
            outputs,
            combined_labels
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == combined_labels
        ).sum().item()

        train_total += combined_labels.size(0)

        loop.set_description(
            f"Epoch [{epoch+1}/{epochs}] "
            f"(LR: {current_lr:.1e})"
        )

    avg_train_loss = (
        running_loss / len(train_loader)
    )

    train_accuracy = (
        100.0 * train_correct / train_total
    )

    print(
        f"\nEpoch {epoch+1}/{epochs} "
        f"Train Loss: {avg_train_loss:.4f} "
        f"| Train Acc: {train_accuracy:.2f}%"
    )

    # =====================================================
    # VALIDATION ON CLEAN DATA
    # =====================================================
    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    val_loop = tqdm(
        val_loader,
        total=len(val_loader),
        leave=False,
        miniters=100
    )

    with torch.no_grad():

        for inputs, labels in val_loop:

            inputs = inputs.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            ).long()

            outputs = model(inputs)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = (
        val_loss / len(val_loader)
    )

    val_accuracy = (
        100.0 * correct / total
    )

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Validation Loss: {avg_val_loss:.4f} "
        f"| Validation Acc: {val_accuracy:.2f}%"
    )

    # Save epoch checkpoint
    torch.save(
        model.state_dict(),
        f"xception_adv_epoch_{epoch+1}.pth"
    )

    scheduler.step(avg_val_loss)
    early_stopper(
        avg_val_loss,
        model
    )

    if early_stopper.early_stop:
        print(
            f"Early stopping triggered at "
            f"epoch {epoch+1}."
        )
        break

# Load best validation model
if early_stopper.best_model_wts is not None:
    print("Loading best model weights...")
    model.load_state_dict(
        early_stopper.best_model_wts
    )

print("\nAdversarial training completed.")

In [ ]:
# Cell 18 - FINAL ART ROBUSTNESS EVALUATION
# This cell proves that ART is actually generating adversarial examples
# and measures Clean vs FGSM vs PGD performance.

model.eval()

# Set this to None if you want to attack the complete test set.
# 256 is a practical starting point for Kaggle/limited GPU time.
ART_EVAL_MAX_SAMPLES = 256

def evaluate_art_attack(
    classifier,
    attack,
    loader,
    max_samples=None,
    attack_name="Attack"
):
    all_clean_preds = []
    all_adv_preds = []
    all_labels = []

    total_seen = 0

    for images, labels in tqdm(
        loader,
        desc=f"ART {attack_name}"
    ):
        remaining = None

        if max_samples is not None:
            remaining = max_samples - total_seen

            if remaining <= 0:
                break

            images = images[:remaining]
            labels = labels[:remaining]

        x_clean = (
            images.cpu()
            .numpy()
            .astype(np.float32)
        )

        y = (
            labels.cpu()
            .numpy()
            .astype(np.int64)
        )

        # Clean prediction
        clean_logits = classifier.predict(
            x_clean,
            batch_size=len(x_clean)
        )

        clean_preds = np.argmax(
            clean_logits,
            axis=1
        )

        # ART adversarial generation
        x_adv = attack.generate(
            x=x_clean,
            y=y
        )

        # Adversarial prediction
        adv_logits = classifier.predict(
            x_adv,
            batch_size=len(x_adv)
        )

        adv_preds = np.argmax(
            adv_logits,
            axis=1
        )

        all_clean_preds.extend(
            clean_preds.tolist()
        )

        all_adv_preds.extend(
            adv_preds.tolist()
        )

        all_labels.extend(
            y.tolist()
        )

        total_seen += len(y)

    labels_np = np.array(all_labels)
    clean_np = np.array(all_clean_preds)
    adv_np = np.array(all_adv_preds)

    clean_acc = np.mean(
        clean_np == labels_np
    )

    adv_acc = np.mean(
        adv_np == labels_np
    )

    attack_success = np.mean(
        (clean_np == labels_np) &
        (adv_np != labels_np)
    )

    return {
        "clean_accuracy": clean_acc,
        "adv_accuracy": adv_acc,
        "attack_success_rate_on_clean_correct": attack_success,
        "labels": labels_np,
        "clean_preds": clean_np,
        "adv_preds": adv_np
    }


print("\n========== FGSM EVALUATION ==========")

fgsm_results = evaluate_art_attack(
    classifier=classifier,
    attack=attack_fgsm,
    loader=test_loader,
    max_samples=ART_EVAL_MAX_SAMPLES,
    attack_name="FGSM"
)

print(
    f"Clean Accuracy : "
    f"{fgsm_results['clean_accuracy'] * 100:.2f}%"
)

print(
    f"FGSM Accuracy  : "
    f"{fgsm_results['adv_accuracy'] * 100:.2f}%"
)

print(
    f"Attack Success : "
    f"{fgsm_results['attack_success_rate_on_clean_correct'] * 100:.2f}%"
)


print("\n========== PGD EVALUATION ==========")

pgd_results = evaluate_art_attack(
    classifier=classifier,
    attack=attack_pgd,
    loader=test_loader,
    max_samples=ART_EVAL_MAX_SAMPLES,
    attack_name="PGD"
)

print(
    f"Clean Accuracy : "
    f"{pgd_results['clean_accuracy'] * 100:.2f}%"
)

print(
    f"PGD Accuracy   : "
    f"{pgd_results['adv_accuracy'] * 100:.2f}%"
)

print(
    f"Attack Success : "
    f"{pgd_results['attack_success_rate_on_clean_correct'] * 100:.2f}%"
)

In [ ]:
# Cell 19 - ART Perturbation Verification + Visualization
# This is additional evidence for your report/viva that ART really
# generated a changed adversarial input.

images, labels = next(iter(test_loader))

# Use one sample for visualization
x_clean = (
    images[0:1]
    .cpu()
    .numpy()
    .astype(np.float32)
)

y = (
    labels[0:1]
    .cpu()
    .numpy()
    .astype(np.int64)
)

# Generate an FGSM example
x_adv = attack_fgsm.generate(
    x=x_clean,
    y=y
)

# Measure perturbation
perturbation = x_adv - x_clean

print("========== ART PERTURBATION CHECK ==========")
print(
    "Maximum absolute perturbation:",
    float(np.max(np.abs(perturbation)))
)

print(
    "Mean absolute perturbation:",
    float(np.mean(np.abs(perturbation)))
)

print(
    "Perturbation is non-zero:",
    bool(np.any(np.abs(perturbation) > 0))
)

# Predictions before/after attack
clean_logits = classifier.predict(x_clean)
adv_logits = classifier.predict(x_adv)

clean_pred = int(np.argmax(clean_logits, axis=1)[0])
adv_pred = int(np.argmax(adv_logits, axis=1)[0])
true_label = int(y[0])

label_names = {
    0: "Real",
    1: "Fake"
}

print(f"True label       : {label_names[true_label]}")
print(f"Clean prediction : {label_names[clean_pred]}")
print(f"FGSM prediction  : {label_names[adv_pred]}")

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

clean_img = (
    np.transpose(x_clean[0], (1, 2, 0))
    * std + mean
)

adv_img = (
    np.transpose(x_adv[0], (1, 2, 0))
    * std + mean
)

# Scale perturbation only for visualization
pert_vis = np.transpose(
    perturbation[0],
    (1, 2, 0)
)

pert_vis = np.abs(pert_vis)
pert_vis = pert_vis / (
    pert_vis.max() + 1e-8
)

clean_img = np.clip(clean_img, 0, 1)
adv_img = np.clip(adv_img, 0, 1)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

axes[0].imshow(clean_img)
axes[0].set_title(
    f"Clean\nPrediction: {label_names[clean_pred]}"
)
axes[0].axis("off")

axes[1].imshow(pert_vis)
axes[1].set_title("FGSM Perturbation\n(amplified)")
axes[1].axis("off")

axes[2].imshow(adv_img)
axes[2].set_title(
    f"Adversarial\nPrediction: {label_names[adv_pred]}"
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 20 - Grad-CAM Explanation
# Grad-CAM is performed AFTER adversarial training and uses the
# final trained model.

# Make sure backbone parameters can receive gradients.
for param in model.backbone.parameters():
    param.requires_grad = True


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.forward_handle = target_layer.register_forward_hook(
            self.save_activation
        )

        self.backward_handle = target_layer.register_full_backward_hook(
            self.save_gradient
        )

    def save_activation(self, module, inputs, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate_heatmap(self, input_tensor):
        self.model.zero_grad(set_to_none=True)

        output = self.model(input_tensor)

        # Choose the model's predicted class.
        target_class = int(
            torch.argmax(output, dim=1).item()
        )

        # Backpropagate only the selected class logit.
        score = output[0, target_class]
        score.backward()

        pooled_gradients = torch.mean(
            self.gradients[0],
            dim=(1, 2)
        )

        activations = self.activations[0].detach()

        weighted_activations = (
            activations *
            pooled_gradients[:, None, None]
        )

        heatmap = torch.mean(
            weighted_activations,
            dim=0
        )

        heatmap = torch.relu(heatmap)

        heatmap = heatmap.cpu().numpy()

        if np.max(heatmap) > 0:
            heatmap /= np.max(heatmap)

        return heatmap, target_class


# Automatically find the last Conv2d layer.
target_layer = None

for module in reversed(
    list(model.backbone.modules())
):
    if isinstance(module, nn.Conv2d):
        target_layer = module
        break

if target_layer is None:
    raise RuntimeError(
        "Could not find a convolutional layer for Grad-CAM."
    )

print("Grad-CAM target layer:", target_layer)

cam = GradCAM(
    model,
    target_layer
)

model.eval()

sample_img, sample_label = next(
    iter(test_loader)
)

sample_img = sample_img[0:1].to(device)

heatmap, predicted_class = cam.generate_heatmap(
    sample_img
)

# Un-normalize image
inv_normalize = transforms.Normalize(
    mean=[
        -0.485 / 0.229,
        -0.456 / 0.224,
        -0.406 / 0.225
    ],
    std=[
        1 / 0.229,
        1 / 0.224,
        1 / 0.225
    ]
)

disp_img = inv_normalize(
    sample_img[0].detach().cpu()
)

disp_img = (
    disp_img.permute(1, 2, 0).numpy()
)

disp_img = np.clip(
    disp_img,
    0,
    1
)

heatmap_resized = cv2.resize(
    heatmap,
    (299, 299)
)

heatmap_colored = cv2.applyColorMap(
    np.uint8(255 * heatmap_resized),
    cv2.COLORMAP_JET
)

heatmap_colored = (
    np.float32(heatmap_colored) / 255
)

heatmap_colored = (
    heatmap_colored[..., ::-1]
)

overlay = (
    heatmap_colored * 0.5
    + disp_img
)

overlay = np.clip(
    overlay,
    0,
    1
)

label_names = {
    0: "Real",
    1: "Fake"
}

true_class = int(sample_label[0].item())

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

axes[0].imshow(disp_img)
axes[0].set_title(
    f"Original | True: {label_names[true_class]}"
)
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title(
    f"Grad-CAM | Predicted: {label_names[predicted_class]}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 21 - FINAL CLEAN TEST EVALUATION + SAVE MODEL
print("Starting Final Evaluation on Clean Test Set...")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for inputs, labels in tqdm(
        test_loader,
        desc="Testing"
    ):
        inputs = inputs.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        ).long()

        outputs = model(inputs)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy().tolist()
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

acc = accuracy_score(
    all_labels,
    all_preds
)

prec = precision_score(
    all_labels,
    all_preds,
    zero_division=0
)

rec = recall_score(
    all_labels,
    all_preds,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_preds,
    zero_division=0
)

print("\n--- Final Xception + ART Adversarial Training Performance ---")
print(f"Clean Accuracy: {acc * 100:.2f}%")
print(f"Precision:      {prec * 100:.2f}%")
print(f"Recall:         {rec * 100:.2f}%")
print(f"F1-Score:       {f1 * 100:.2f}%")

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title(
    "Xception Deepfake Detection Confusion Matrix"
)
plt.colorbar()

plt.xticks(
    [0, 1],
    ["Predicted Real", "Predicted Fake"]
)

plt.yticks(
    [0, 1],
    ["Actually Real", "Actually Fake"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

# Save final trained model
torch.save(
    model.state_dict(),
    "xception_deepfake_detector_adversarial_final.pth"
)

print(
    "Final adversarially trained model saved successfully."
)